# 01 - Veri Seti Tanıtımı

**Sorumlu:** Melike Türkdal

**Ödevdeki karşılığı:** 1. Veri Seti Seçimi ve Tanıtımı

In [1]:
# KURULUM: bu hucreyi her oturumun basinda bir kez calistir.
# Repoyu Colab'e indirir; boylece src/ortak.py kullanilabilir.
import os
REPO_URL = "https://github.com/zeyneppbaran/ml-proje.git"  # TODO: kendi repo adresiniz
if not os.path.exists("/content/ml-proje"):
    !git clone {REPO_URL} /content/ml-proje
else:
    !git -C /content/ml-proje pull
%cd /content/ml-proje

Cloning into '/content/ml-proje'...
remote: Enumerating objects: 85, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 85 (delta 51), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (85/85), 24.26 KiB | 4.85 MiB/s, done.
Resolving deltas: 100% (51/51), done.
/content/ml-proje


In [3]:
%pip install ucimlrepo

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#from src.ortak import veri_yukle

HEDEF_SUTUN = "y"
VERI_YOLU = "data/bank.csv"


def veri_yukle():
    """Ham veriyi yukler. Hedef sutun yes=1, no=0 olarak kodlanir.

    Ilk calistirmada veri UCI'den indirilip data/ klasorune kaydedilir,
    sonraki calistirmalarda yerel kopyadan okunur.
    """
    if os.path.exists(VERI_YOLU):
        df = pd.read_csv(VERI_YOLU)
    else:
        from ucimlrepo import fetch_ucirepo
        bank = fetch_ucirepo(id=222)
        df = pd.concat([bank.data.features, bank.data.targets], axis=1)
        os.makedirs(os.path.dirname(VERI_YOLU), exist_ok=True)
        df.to_csv(VERI_YOLU, index=False)

    df[HEDEF_SUTUN] = df[HEDEF_SUTUN].map({"yes": 1, "no": 0})
    return df

df = veri_yukle()
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN,0
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN,0
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN,0
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN,0
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN,0


## B1. Kaynak ve amaç

In [ ]:
# kod buraya

**Yorum:** (bu çıktı bize ne söylüyor?)

## B2. Veri setinin yapısı (boyut, özellikler, veri türleri)

In [7]:
df.shape

(45211, 17)

**Yorum:** Çalışmada kullanılan veri seti 45.211 gözlem ve 17 sütundan oluşmaktadır. Sütunlardan 16'sı açıklayıcı değişken (özellik), 1'i hedef değişkendir (y).

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   age          45211 non-null  int64 
 1   job          44923 non-null  object
 2   marital      45211 non-null  object
 3   education    43354 non-null  object
 4   default      45211 non-null  object
 5   balance      45211 non-null  int64 
 6   housing      45211 non-null  object
 7   loan         45211 non-null  object
 8   contact      32191 non-null  object
 9   day_of_week  45211 non-null  int64 
 10  month        45211 non-null  object
 11  duration     45211 non-null  int64 
 12  campaign     45211 non-null  int64 
 13  pdays        45211 non-null  int64 
 14  previous     45211 non-null  int64 
 15  poutcome     8252 non-null   object
 16  y            45211 non-null  int64 
dtypes: int64(8), object(9)
memory usage: 5.9+ MB


**yorum:**
* Özelliklerin 7'si sayısal, 9'u kategorik türdedir. Hedef değişken y, orijinal veride yes/no değerlerini içeren kategorik bir değişkendir. Modelleme için 0/1 olarak kodlandığından int64 türünde görünmektedir.
* 4 kategorik sütunda eksik değer bulunmaktadır. Diğer 13 sütun eksiksizdir. Orijinal veri setinde bu değerler "unknown" olarak kodlanmış olup, kullanılan yükleme yöntemiyle NaN olarak aktarılmıştır

### Veri Sözlüğü

In [17]:
sozluk = pd.DataFrame({
    "tur" : df.dtypes.astype(str),
    "benzersiz": df.nunique(),
    "eksik": df.isna().sum(),
    "ornek_degerler": [df[c].dropna().unique()[:5].tolist() for c in df.columns],
})

sozluk

,tur,benzersiz,eksik,ornek_degerler
age,int64,77,0,"[58, 44, 33, 47, 35]"
job,object,11,288,"[management, technician, entrepreneur, blue-co..."
marital,object,3,0,"[married, single, divorced]"
education,object,3,1857,"[tertiary, secondary, primary]"
default,object,2,0,"[no, yes]"
balance,int64,7168,0,"[2143, 29, 2, 1506, 1]"
housing,object,2,0,"[yes, no]"
loan,object,2,0,"[no, yes]"
contact,object,2,13020,"[cellular, telephone]"
day_of_week,int64,31,0,"[5, 6, 7, 8, 9]"


In [18]:
display(df.describe().T)

for col in df.select_dtypes(include="number").columns:
  print(col, "-> en sık 3 değer:", df[col].value_counts().head(3).to_dict())

,count,mean,std,min,25%,50%,75%,max
age,45211.0,40.936210,10.618762,18.0,33.0,39.0,48.0,95.0
balance,45211.0,1362.272058,3044.765829,-8019.0,72.0,448.0,1428.0,102127.0
day_of_week,45211.0,15.806419,8.322476,1.0,8.0,16.0,21.0,31.0
duration,45211.0,258.163080,257.527812,0.0,103.0,180.0,319.0,4918.0
campaign,45211.0,2.763841,3.098021,1.0,1.0,2.0,3.0,63.0
pdays,45211.0,40.197828,100.128746,-1.0,-1.0,-1.0,-1.0,871.0
previous,45211.0,0.580323,2.303441,0.0,0.0,0.0,0.0,275.0
y,45211.0,0.116985,0.321406,0.0,0.0,0.0,0.0,1.0


age -> en sık 3 değer: {32: 2085, 31: 1996, 33: 1972}
balance -> en sık 3 değer: {0: 3514, 1: 195, 2: 156}
day_of_week -> en sık 3 değer: {20: 2752, 18: 2308, 21: 2026}
duration -> en sık 3 değer: {124: 188, 90: 184, 89: 177}
campaign -> en sık 3 değer: {1: 17544, 2: 12505, 3: 5521}
pdays -> en sık 3 değer: {-1: 36954, 182: 167, 92: 147}
previous -> en sık 3 değer: {0: 36954, 1: 2772, 2: 2106}
y -> en sık 3 değer: {0: 39922, 1: 5289}


### Eksik veri kontrolü

In [14]:
# 1. Standart NaN kontrolü
eksik = df.isna().sum()
eksik_oran = df.isna().mean() * 100
display(pd.DataFrame({"adet": eksik, "oran_%": eksik_oran.round(2)})[eksik>0])

,adet,oran_%
job,288,0.64
education,1857,4.11
contact,13020,28.80
poutcome,36959,81.75


In [15]:
# 2. metin tabanlı gizli eksiklik (unknown) kontrolü
for col in df.select_dtypes(include="object").columns:
    unknown_sayisi = (df[col].str.strip().str.lower() == "unknown").sum()
    print(f"{col}: {unknown_sayisi if unknown_sayisi > 0 else 'unknown değer yok'}")


job: unknown değer yok
marital: unknown değer yok
education: unknown değer yok
default: unknown değer yok
housing: unknown değer yok
loan: unknown değer yok
contact: unknown değer yok
month: unknown değer yok
poutcome: unknown değer yok


In [ ]:
# sayısal yer tutucu kontrolü (-1)
for col in df.select_dtype(include="number").columns

In [20]:
df["month"].unique()

array(['may', 'jun', 'jul', 'aug', 'oct', 'nov', 'dec', 'jan', 'feb',
       'mar', 'apr', 'sep'], dtype=object)

## B3. Hedef değişken

In [ ]:
# kod buraya

**Yorum:** (bu çıktı bize ne söylüyor?)

## B4. Sınıf dağılımı

In [ ]:
# kod buraya

**Yorum:** (bu çıktı bize ne söylüyor?)

## B5. Anormal ve problemli veriler

In [ ]:
# kod buraya

**Yorum:** (bu çıktı bize ne söylüyor?)